In [2]:
# ===============================
# MOUNT DRIVE (SAFE)
# ===============================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# ===============================
# IMPORTS
# ===============================
import cv2
import numpy as np
import os
import pandas as pd
import time
import tracemalloc
import platform
import multiprocessing

from scipy.special import gamma

from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import mean_squared_error as mse
from skimage.measure import shannon_entropy

np.random.seed(42)

# ===============================
# SYSTEM CONFIGURATION
# ===============================
print("===================================")
print("SYSTEM CONFIGURATION")
print("===================================")
print("Processor:", platform.processor())
print("CPU Cores:", multiprocessing.cpu_count())
print("Platform:", platform.platform())

# ===============================
# PATHS
# ===============================
image_folder = "/content/drive/MyDrive/Colab Notebooks/data/low_quality_images"

output_folder = "/content/drive/MyDrive/Colab Notebooks/results/output_CLAHE/FINAL_FIXED"

results_folder = "/content/drive/MyDrive/Colab Notebooks/results/output_CLAHE"

os.makedirs(output_folder, exist_ok=True)
os.makedirs(results_folder, exist_ok=True)

# ===============================
# METRICS
# ===============================
def compute_metrics(original, processed):

    original = original.astype(np.float32)
    processed = processed.astype(np.float32)

    ssim_val = ssim(
        original,
        processed,
        data_range=255
    )

    psnr_val = psnr(
        original,
        processed,
        data_range=255
    )

    mse_val = mse(
        original,
        processed
    )

    mae_val = np.mean(
        np.abs(original - processed)
    )

    entropy_val = shannon_entropy(
        processed
    )

    edges_orig = cv2.Canny(
        original.astype(np.uint8),
        100,
        200
    )

    edges_proc = cv2.Canny(
        processed.astype(np.uint8),
        100,
        200
    )

    if np.sum(edges_orig) == 0:
        epi_val = 0
    else:
        epi_val = (
            np.sum(edges_orig & edges_proc)
            /
            np.sum(edges_orig)
        )

    return (
        ssim_val,
        psnr_val,
        mse_val,
        mae_val,
        entropy_val,
        epi_val
    )

# ===============================
# RUNTIME EFFICIENCY
# ===============================
def runtime_efficiency(runtime):

    if runtime == 0:
        return 0

    return 1 / runtime

# ===============================
# ENHANCEMENT FUNCTION
# ===============================
def enhance_image(
    img,
    clip,
    tile,
    alpha,
    sigma
):

    clahe = cv2.createCLAHE(
        clipLimit=float(clip),
        tileGridSize=(int(tile), int(tile))
    )

    cl = clahe.apply(img)

    blur = cv2.GaussianBlur(
        cl,
        (0, 0),
        sigma
    )

    sharp = cv2.addWeighted(
        cl,
        1 + alpha,
        blur,
        -alpha,
        0
    )

    return np.clip(
        sharp,
        0,
        255
    ).astype(np.uint8)

# ===============================
# EDGE METRIC
# ===============================
def tenengrad(img):

    gx = cv2.Sobel(
        img,
        cv2.CV_64F,
        1,
        0,
        ksize=3
    )

    gy = cv2.Sobel(
        img,
        cv2.CV_64F,
        0,
        1,
        ksize=3
    )

    return np.mean(
        gx**2 + gy**2
    )

# ===============================
# FITNESS FUNCTION
# ===============================
def get_fitness(img):

    def f(p):

        clip, tile, alpha, sigma = p

        enhanced = enhance_image(
            img,
            clip,
            tile,
            alpha,
            sigma
        )

        edge = tenengrad(
            enhanced
        )

        noise = mse(
            img,
            enhanced
        )

        return (
            0.5 * edge
            -
            0.5 * noise
        )

    return f

# ===============================
# ICS CLASS
# ===============================
class ICS:

    def __init__(
        self,
        fitness,
        bounds,
        n=10,
        pa=0.25,
        beta=1.5,
        iters=8
    ):

        self.fit = fitness
        self.bounds = np.array(bounds)

        self.n = n
        self.pa = pa
        self.beta = beta
        self.iters = iters

        self.dim = len(bounds)

        self.pop = np.random.uniform(
            self.bounds[:,0],
            self.bounds[:,1],
            (n,self.dim)
        )

        self.fvals = np.array([
            self.fit(x)
            for x in self.pop
        ])

    def levy(self):

        b = self.beta

        sigma = (
            gamma(1+b)
            *
            np.sin(np.pi*b/2)
            /
            (
                gamma((1+b)/2)
                *
                b
                *
                2**((b-1)/2)
            )
        )**(1/b)

        u = np.random.normal(
            0,
            sigma,
            self.dim
        )

        v = np.random.normal(
            0,
            1,
            self.dim
        )

        return u/(np.abs(v)**(1/b))

    def clip(self, x):

        return np.clip(
            x,
            self.bounds[:,0],
            self.bounds[:,1]
        )

    def run(self):

        for _ in range(self.iters):

            for i in range(self.n):

                new = self.clip(
                    self.pop[i]
                    +
                    0.03*self.levy()
                )

                fnew = self.fit(new)

                j = np.random.randint(
                    self.n
                )

                if fnew > self.fvals[j]:

                    self.pop[j] = new
                    self.fvals[j] = fnew

            for i in range(self.n):

                if np.random.rand() < self.pa:

                    self.pop[i] = np.random.uniform(
                        self.bounds[:,0],
                        self.bounds[:,1],
                        self.dim
                    )

                    self.fvals[i] = self.fit(
                        self.pop[i]
                    )

        return self.pop[
            np.argmax(self.fvals)
        ]

# ===============================
# LOAD IMAGES
# ===============================
files = [
    f for f in os.listdir(image_folder)
    if f.lower().endswith(
        (".jpg",".png",".jpeg")
    )
][:25]

records = []

bounds = [
    (1.0, 3.0),
    (6, 10),
    (0.3, 1.0),
    (0.5, 1.2)
]

# ===============================
# MAIN LOOP
# ===============================
for file in files:

    img = cv2.imread(
        os.path.join(
            image_folder,
            file
        )
    )

    gray = cv2.resize(
        cv2.cvtColor(
            img,
            cv2.COLOR_BGR2GRAY
        ),
        (256,256)
    )

    gray = cv2.fastNlMeansDenoising(
        gray,
        None,
        10,
        7,
        21
    )

    # ==========================
    # PERFORMANCE TRACKING
    # ==========================
    tracemalloc.start()

    start_time = time.perf_counter()

    ics = ICS(
        get_fitness(gray),
        bounds
    )

    best = ics.run()

    enhanced = enhance_image(
        gray,
        *best
    )

    end_time = time.perf_counter()

    current_mem, peak_mem = tracemalloc.get_traced_memory()

    tracemalloc.stop()

    execution_time = (
        end_time - start_time
    )

    memory_usage = (
        peak_mem / (1024 * 1024)
    )

    efficiency = runtime_efficiency(
        execution_time
    )

    cv2.imwrite(
        os.path.join(
            output_folder,
            file
        ),
        enhanced
    )

    s, p, m, mae, ent, epi = compute_metrics(
        gray,
        enhanced
    )

    records.append([
        file,
        s,
        p,
        m,
        mae,
        ent,
        epi,
        execution_time,
        memory_usage,
        efficiency
    ])

    print(
        f"Processed: {file}"
    )

# ===============================
# RESULTS DATAFRAME
# ===============================
df = pd.DataFrame(
    records,
    columns=[
        "Image",
        "SSIM",
        "PSNR",
        "MSE",
        "MAE",
        "Entropy",
        "EPI",
        "Execution_Time_sec",
        "Memory_MB",
        "Runtime_Efficiency"
    ]
)

# ===============================
# SAVE RESULTS
# ===============================
df.to_csv(
    os.path.join(results_folder, "final_results.csv"),
    index=False
)

# ===============================
# PERFORMANCE SUMMARY
# ===============================
performance_summary = pd.DataFrame({

    "Metric":[
        "Average Execution Time (s)",
        "Average Memory Usage (MB)",
        "Average Runtime Efficiency"
    ],

    "Value":[
        df["Execution_Time_sec"].mean(),
        df["Memory_MB"].mean(),
        df["Runtime_Efficiency"].mean()
    ]
})

performance_summary.to_csv(
    os.path.join(results_folder, "performance_summary.csv"),
    index=False
)

system_info = pd.DataFrame({
    "Parameter": [
        "Processor",
        "CPU Cores",
        "Platform"
    ],
    "Value": [
        platform.processor(),
        multiprocessing.cpu_count(),
        platform.platform()
    ]
})

system_info.to_csv(
    os.path.join(results_folder, "system_info.csv"),
    index=False
)

# ===============================
# DISPLAY RESULTS
# ===============================
print("\n===================================")
print("AVERAGE IMAGE QUALITY METRICS")
print("===================================")
print(df.mean(numeric_only=True))

print("\n===================================")
print("COMPUTATIONAL PERFORMANCE")
print("===================================")
print(performance_summary)

print("\nFINAL RUN DONE 🚀")

Mounted at /content/drive
SYSTEM CONFIGURATION
Processor: x86_64
CPU Cores: 2
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
Processed: 00 (74).jpg
Processed: 00 (139).jpg
Processed: 00 (84).jpg
Processed: 00 (137).jpg
Processed: 00 (59).jpg
Processed: 00 (112).jpg
Processed: 00 (79).jpg
Processed: 00 (57).jpg
Processed: 00 (132).jpg
Processed: 00 (81).jpg
Processed: 00 (111).jpg
Processed: 00 (109).jpg
Processed: 00 (106).jpg
Processed: 00 (116).jpg
Processed: 00 (131).jpg
Processed: 00 (78).jpg
Processed: 00 (77).jpg
Processed: 00 (141).jpg
Processed: 00 (138).jpg
Processed: 00 (80).jpg
Processed: 00 (54).jpg
Processed: 00 (113).jpg
Processed: 00 (134).jpg
Processed: 00 (133).jpg
Processed: 00 (107).jpg

AVERAGE IMAGE QUALITY METRICS
SSIM                     0.772180
PSNR                    17.200446
MSE                   1274.141116
MAE                     25.978155
Entropy                  7.711254
EPI                      0.841974
Execution_Time_sec       0.290417
Memory_MB       